# Autocomplete and Autocorrect Data Analytics

**Oasis Infobyte Data Analytics Internship — Level 2, Task 5**

Classical NLP approaches for next-word prediction and spelling correction.


## 1. Imports and Configuration


In [ ]:
import re
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from spellchecker import SpellChecker

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

CORPUS_PATH = "corpus.txt"

# Download these resources once if required:
# import nltk
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")


## 2. Load the Text Corpus


In [ ]:
with open(CORPUS_PATH, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

print("Raw characters:", len(raw_text))
print(raw_text[:1000])


## 3. NLP Preprocessing


In [ ]:
text_lower = raw_text.lower()

# Keep alphabetic tokens and apostrophe-free word forms for a simple,
# reproducible educational corpus.
tokens_raw = re.findall(r"[a-z]+", text_lower)

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    import nltk
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

tokens = [
    token for token in tokens_raw
    if token not in stop_words
]

print("Raw tokens:", len(tokens_raw))
print("Tokens after stopword removal:", len(tokens))
print("Vocabulary size:", len(set(tokens)))

display(pd.DataFrame(
    Counter(tokens).most_common(20),
    columns=["Word", "Frequency"]
))


## 4. Top 20 Most Frequent Words


In [ ]:
top20 = pd.DataFrame(
    Counter(tokens).most_common(20),
    columns=["Word", "Frequency"]
)

plt.figure(figsize=(11, 7))
sns.barplot(
    data=top20,
    x="Frequency",
    y="Word"
)
plt.title("Top 20 Most Frequent Words")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()


## 5. Train/Test Split


In [ ]:
# Keep sequence order so the held-out tail can be used for next-word evaluation.
split_index = int(len(tokens) * 0.8)

train_tokens = tokens[:split_index]
test_tokens = tokens[split_index:]

print("Training tokens:", len(train_tokens))
print("Test tokens:", len(test_tokens))


## 6. Build Bigram and Trigram Models


In [ ]:
bigram_counts = defaultdict(Counter)
trigram_counts = defaultdict(Counter)

for i in range(len(train_tokens) - 1):
    bigram_counts[train_tokens[i]][train_tokens[i + 1]] += 1

for i in range(len(train_tokens) - 2):
    context = (train_tokens[i], train_tokens[i + 1])
    trigram_counts[context][train_tokens[i + 2]] += 1

def bigram_predict(prefix, top_k=3):
    prefix = prefix.lower().strip()
    candidates = bigram_counts.get(prefix, Counter())
    return [word for word, _ in candidates.most_common(top_k)]

def trigram_predict(prefix, top_k=3):
    parts = prefix.lower().split()

    if len(parts) < 2:
        return []

    context = (parts[-2], parts[-1])
    candidates = trigram_counts.get(context, Counter())
    return [word for word, _ in candidates.most_common(top_k)]


## 7. Test Autocomplete on at Least 10 Prefixes


In [ ]:
# Use frequent training words to make the examples reproducible.
test_prefixes = [
    word for word, count in Counter(train_tokens).most_common(30)
    if word in bigram_counts
][:10]

autocomplete_results = []

for prefix in test_prefixes:
    autocomplete_results.append({
        "Prefix": prefix,
        "Bigram Top 3": ", ".join(bigram_predict(prefix, 3))
    })

# Build ten two-word contexts for trigram testing.
trigram_contexts = [
    " ".join(train_tokens[i:i+2])
    for i in range(len(train_tokens)-2)
    if (train_tokens[i], train_tokens[i+1]) in trigram_counts
]

trigram_contexts = list(dict.fromkeys(trigram_contexts))[:10]

for context in trigram_contexts:
    autocomplete_results.append({
        "Prefix": context,
        "Bigram Top 3": "",
        "Trigram Top 3": ", ".join(trigram_predict(context, 3))
    })

display(pd.DataFrame(autocomplete_results))


## 8. Autocomplete Precision and Recall


In [ ]:
def evaluate_bigram(test_sequence, top_k=3):
    hits = 0
    evaluated = 0

    for i in range(len(test_sequence) - 1):
        prefix = test_sequence[i]
        actual = test_sequence[i + 1]

        predictions = bigram_predict(prefix, top_k)

        if predictions:
            evaluated += 1
            if actual in predictions:
                hits += 1

    if evaluated == 0:
        return {"precision": np.nan, "recall": np.nan, "hits": 0, "evaluated": 0}

    # For top-k next-word retrieval, each evaluated prefix generates k candidate
    # predictions and one true next word.
    precision = hits / (evaluated * top_k)
    recall = hits / evaluated

    return {
        "precision": precision,
        "recall": recall,
        "hits": hits,
        "evaluated": evaluated
    }

def evaluate_trigram(test_sequence, top_k=3):
    hits = 0
    evaluated = 0

    for i in range(len(test_sequence) - 2):
        context = (test_sequence[i], test_sequence[i + 1])
        actual = test_sequence[i + 2]

        predictions = trigram_predict(" ".join(context), top_k)

        if predictions:
            evaluated += 1
            if actual in predictions:
                hits += 1

    if evaluated == 0:
        return {"precision": np.nan, "recall": np.nan, "hits": 0, "evaluated": 0}

    precision = hits / (evaluated * top_k)
    recall = hits / evaluated

    return {
        "precision": precision,
        "recall": recall,
        "hits": hits,
        "evaluated": evaluated
    }

bigram_metrics = evaluate_bigram(test_tokens)
trigram_metrics = evaluate_trigram(test_tokens)

autocomplete_metrics = pd.DataFrame([
    {"Approach": "Bigram", **bigram_metrics},
    {"Approach": "Trigram", **trigram_metrics}
])

display(autocomplete_metrics)


## 9. Autocorrect — Prepare Vocabulary


In [ ]:
vocabulary = set(train_tokens)

# Limit the candidate vocabulary to reasonably common words. This keeps
# custom edit-distance search computationally practical.
word_frequency = Counter(train_tokens)
candidate_vocabulary = {
    word for word, count in word_frequency.items()
    if count >= 2
}

print("Full vocabulary:", len(vocabulary))
print("Candidate vocabulary:", len(candidate_vocabulary))


## 10. Custom Levenshtein Distance


In [ ]:
def levenshtein_distance(a, b):
    if a == b:
        return 0

    if len(a) < len(b):
        a, b = b, a

    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, start=1):
        current = [i]

        for j, char_b in enumerate(b, start=1):
            insert_cost = current[j - 1] + 1
            delete_cost = previous[j] + 1
            replace_cost = previous[j - 1] + (char_a != char_b)

            current.append(
                min(insert_cost, delete_cost, replace_cost)
            )

        previous = current

    return previous[-1]

def custom_autocorrect(word, vocabulary=candidate_vocabulary, max_candidates=5):
    word = word.lower()

    if word in vocabulary:
        return word

    # First use a cheap length filter.
    nearby = [
        candidate for candidate in vocabulary
        if abs(len(candidate) - len(word)) <= 2
    ]

    if not nearby:
        nearby = list(vocabulary)

    scored = []
    for candidate in nearby:
        distance = levenshtein_distance(word, candidate)
        scored.append((distance, -word_frequency[candidate], candidate))

    scored.sort()
    return scored[0][2] if scored else word


## 11. PySpellChecker Approach


In [ ]:
spell = SpellChecker()

def spellchecker_autocorrect(word):
    word = word.lower()

    if word in vocabulary:
        return word

    correction = spell.correction(word)
    return correction if correction is not None else word


## 12. Twenty Deliberately Misspelled Test Words


In [ ]:
# The intended words are deliberately paired with common typo patterns.
misspelled_pairs = [
    ("recieve", "receive"),
    ("teh", "the"),
    ("adress", "address"),
    ("definately", "definitely"),
    ("seperate", "separate"),
    ("occured", "occurred"),
    ("becuase", "because"),
    ("enviroment", "environment"),
    ("wierd", "weird"),
    ("goverment", "government"),
    ("sucess", "success"),
    ("tommorow", "tomorrow"),
    ("begining", "beginning"),
    ("responsibilty", "responsibility"),
    ("availble", "available"),
    ("langauge", "language"),
    ("reccomend", "recommend"),
    ("experiance", "experience"),
    ("acheive", "achieve"),
    ("calender", "calendar")
]

autocorrect_rows = []

for misspelled, intended in misspelled_pairs:
    spell_prediction = spellchecker_autocorrect(misspelled)
    custom_prediction = custom_autocorrect(misspelled)

    autocorrect_rows.append({
        "Misspelled": misspelled,
        "Intended": intended,
        "SpellChecker": spell_prediction,
        "SpellChecker_Correct": spell_prediction == intended,
        "Levenshtein": custom_prediction,
        "Levenshtein_Correct": custom_prediction == intended
    })

autocorrect_results = pd.DataFrame(autocorrect_rows)
display(autocorrect_results)


## 13. Autocorrect Accuracy, Precision and Recall


In [ ]:
def binary_correction_metrics(results, prediction_col, target_col="Intended"):
    tp = (results[prediction_col] == results[target_col]).sum()
    fp = (results[prediction_col] != results[target_col]).sum()
    fn = fp

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    accuracy = tp / len(results) if len(results) else 0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "correct": int(tp),
        "incorrect": int(fp)
    }

autocorrect_metrics = pd.DataFrame([
    {
        "Approach": "PySpellChecker",
        **binary_correction_metrics(
            autocorrect_results,
            "SpellChecker"
        )
    },
    {
        "Approach": "Custom Levenshtein",
        **binary_correction_metrics(
            autocorrect_results,
            "Levenshtein"
        )
    }
])

display(autocorrect_metrics)


## 14. Autocorrect Confusion Matrix — Correct vs Incorrect


In [ ]:
def outcome_matrix(results, prediction_col):
    return pd.DataFrame(
        [[
            (results[prediction_col] == results["Intended"]).sum(),
            (results[prediction_col] != results["Intended"]).sum()
        ]],
        index=["Predicted correction"],
        columns=["Correct", "Incorrect"]
    )

spell_outcome = outcome_matrix(autocorrect_results, "SpellChecker")
lev_outcome = outcome_matrix(autocorrect_results, "Levenshtein")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(
    spell_outcome,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    ax=axes[0]
)
axes[0].set_title("PySpellChecker Outcome")
axes[0].set_xlabel("Outcome")
axes[0].set_ylabel("")

sns.heatmap(
    lev_outcome,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    ax=axes[1]
)
axes[1].set_title("Custom Levenshtein Outcome")
axes[1].set_xlabel("Outcome")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


## 15. Algorithm Comparison


In [ ]:
comparison = pd.concat([
    autocomplete_metrics.assign(Task="Autocomplete"),
    autocorrect_metrics.assign(Task="Autocorrect")
], ignore_index=True)

display(comparison)


## 16. Discussion — Production Limitations


In [ ]:
limitations = [
    "Frequency-only models do not understand sentence meaning or long-range context.",
    "The vocabulary is limited to the supplied corpus and may miss new names, slang and technical terms.",
    "Simple edit distance treats all character substitutions similarly even though keyboard typos have different probabilities.",
    "The autocorrect benchmark uses only 20 deliberately misspelled words, so results are illustrative rather than production benchmarks.",
    "Production systems can use personalized history, keyboard geometry, language models, multilingual context and large-scale ranking systems.",
    "Latency, memory usage and continual vocabulary updates are important constraints for mobile keyboards."
]

for i, item in enumerate(limitations, 1):
    print(f"{i}. {item}")


## 17. Conclusion


In [ ]:
best_autocomplete = autocomplete_metrics.sort_values(
    "recall", ascending=False
).iloc[0]

best_autocorrect = autocorrect_metrics.sort_values(
    "accuracy", ascending=False
).iloc[0]

print(
    f"Autocomplete comparison: {best_autocomplete['Approach']} has the "
    f"higher measured recall on the supplied held-out corpus."
)

print(
    f"Autocorrect comparison: {best_autocorrect['Approach']} has the "
    f"higher measured accuracy on the 20-word test set."
)

print(
    "These results are corpus- and test-set-dependent. A production keyboard "
    "would require substantially richer contextual and personalization models."
)


## 18. Save Results


In [ ]:
top20.to_csv("top_20_words.csv", index=False)
autocomplete_metrics.to_csv("autocomplete_metrics.csv", index=False)
autocorrect_results.to_csv("autocorrect_test_results.csv", index=False)
autocorrect_metrics.to_csv("autocorrect_metrics.csv", index=False)

print("Analysis outputs saved.")
